## **Comparing Machine Learning and Statistical Models for FIFA World Cup Prediction**

Everything is fit on 358 real international matches: every game from the 2010–2022 World Cups (256 matches) plus the 2020 and 2024 European Championships (102), pulled from the openfootball project, specifically its worldcup.json and euro.json datasets, which are dedicated to the public domain. The classifiers learn a mapping from match features to results on these games; the rating systems are computed directly from the results graph. The field is the real, confirmed 2026 draw, 48 teams, 12 groups.

In [ ]:
#packages
!pip -q install pandas numpy matplotlib seaborn scikit-learn xgboost requests

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import os
import re

from collections import Counter

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries successfully imported.")

Libraries successfully imported.


In [ ]:
#project configuration

RANDOM_STATE = 42

DATA_DIR = "/content/data"
RAW_DIR = f"{DATA_DIR}/raw"
PROCESSED_DIR = f"{DATA_DIR}/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Project directories created.")

Project directories created.


In [ ]:
#URLs for the openfootball datasets

# OpenFootball JSON repositories
BASE_WC = "https://raw.githubusercontent.com/openfootball/worldcup.json/master"
BASE_EURO = "https://raw.githubusercontent.com/openfootball/euro.json/master"

WORLD_CUPS = [2010, 2014, 2018, 2022]
EUROS = [2020, 2024]

print("World Cups:", WORLD_CUPS)
print("European Championships:", EUROS)

World Cups: [2010, 2014, 2018, 2022]
European Championships: [2020, 2024]


# **WorldCup Data**

---



In [ ]:
url = f"{BASE_WC}/2022/worldcup.json"

response = requests.get(url)

print("Status code:", response.status_code)
print("File size:", len(response.text), "characters")

Status code: 200
File size: 25297 characters


In [ ]:
wc2022 = response.json()

print(type(wc2022))
print(wc2022.keys())

<class 'dict'>
dict_keys(['name', 'matches'])


In [ ]:
wc2022["matches"][0]

{'round': 'Matchday 1',
 'date': '2022-11-20',
 'time': '19:00',
 'team1': 'Qatar',
 'team2': 'Ecuador',
 'score': {'ft': [0, 2], 'ht': [0, 2]},
 'goals1': [],
 'goals2': [{'name': 'Enner Valencia', 'minute': '16', 'penalty': True},
  {'name': 'Enner Valencia', 'minute': '31'}],
 'group': 'Group A',
 'ground': 'Al Bayt Stadium, Al Khor'}

In [ ]:
def load_openfootball_tournament(url, tournament_name, year):
    """
    Download and convert an OpenFootball JSON tournament
    into a match-level DataFrame.

    Handles both OpenFootball score formats:
        [goals1, goals2]
    and
        {"ft": [...], "et": [...], "p": [...]}
    """

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    matches = []

    for match in data["matches"]:

        score = match.get("score")

        # Skip matches without a score
        if score is None:
            continue

        # ---------------------------------------
        # FORMAT 1: simple score [team1, team2]
        # ---------------------------------------
        if isinstance(score, list):

            if len(score) != 2:
                continue

            goals1 = score[0]
            goals2 = score[1]

            ft_goals1 = goals1
            ft_goals2 = goals2

            pen1 = None
            pen2 = None

        elif isinstance(score, dict):

            ft_score = score.get("ft")
            et_score = score.get("et")
            pen_score = score.get("p")

            if ft_score is None or len(ft_score) != 2:
                continue

            final_score = (
                et_score
                if et_score is not None
                else ft_score
            )

            goals1 = final_score[0]
            goals2 = final_score[1]

            ft_goals1 = ft_score[0]
            ft_goals2 = ft_score[1]

            if pen_score is not None:
                pen1 = pen_score[0]
                pen2 = pen_score[1]
            else:
                pen1 = None
                pen2 = None

        else:
            continue

        if pen1 is not None:

            if pen1 > pen2:
                result = "team1_win"
            else:
                result = "team2_win"

        elif goals1 > goals2:
            result = "team1_win"

        elif goals1 < goals2:
            result = "team2_win"

        else:
            result = "draw"

        matches.append({
            "date": match.get("date"),
            "tournament": tournament_name,
            "year": year,
            "round": match.get("round"),

            "team1": match.get("team1"),
            "team2": match.get("team2"),

            "goals1": goals1,
            "goals2": goals2,

            "ft_goals1": ft_goals1,
            "ft_goals2": ft_goals2,

            "pen1": pen1,
            "pen2": pen2,

            "result": result,

            "group": match.get("group"),
            "ground": match.get("ground")
        })

    return pd.DataFrame(matches)

In [ ]:
test_2022 = load_openfootball_tournament(
    f"{BASE_WC}/2022/worldcup.json",
    "World Cup",
    2022
)

print("Matches:", len(test_2022))

display(test_2022.head())

Matches: 64


,date,tournament,year,round,team1,team2,goals1,goals2,ft_goals1,ft_goals2,pen1,pen2,result,group,ground
0,2022-11-20,World Cup,2022,Matchday 1,Qatar,Ecuador,0,2,0,2,NaN,NaN,team2_win,Group A,"Al Bayt Stadium, Al Khor"
1,2022-11-21,World Cup,2022,Matchday 2,Senegal,Netherlands,0,2,0,2,NaN,NaN,team2_win,Group A,"Al Thumama Stadium, Doha"
2,2022-11-25,World Cup,2022,Matchday 6,Qatar,Senegal,1,3,1,3,NaN,NaN,team2_win,Group A,"Al Thumama Stadium, Doha"
3,2022-11-25,World Cup,2022,Matchday 6,Netherlands,Ecuador,1,1,1,1,NaN,NaN,draw,Group A,"Khalifa International Stadium, Al Rayyan"
4,2022-11-29,World Cup,2022,Matchday 10,Ecuador,Senegal,1,2,1,2,NaN,NaN,team2_win,Group A,"Khalifa International Stadium, Al Rayyan"


In [ ]:
world_cup_dfs = []

for year in WORLD_CUPS:

    url = f"{BASE_WC}/{year}/worldcup.json"

    df = load_openfootball_tournament(
        url,
        "World Cup",
        year
    )

    print(f"{year}: {len(df)} matches")

    world_cup_dfs.append(df)

world_cups = pd.concat(
    world_cup_dfs,
    ignore_index=True
)

print("\nTotal World Cup matches:", len(world_cups))

2010: 64 matches
2014: 64 matches
2018: 64 matches
2022: 64 matches

Total World Cup matches: 256


# **EuroCup Data**

In [ ]:
euro_dfs = []

for year in EUROS:

    url = f"{BASE_EURO}/{year}/euro.json"

    df = load_openfootball_tournament(
        url,
        "European Championship",
        year
    )

    print(f"Euro {year}: {len(df)} matches")

    euro_dfs.append(df)

euros = pd.concat(
    euro_dfs,
    ignore_index=True
)

print("\nTotal Euro matches:", len(euros))

Euro 2020: 51 matches
Euro 2024: 51 matches

Total Euro matches: 102


# **Combined Data**

In [ ]:
matches_men = pd.concat(
    [world_cups, euros],
    ignore_index=True
)

print("Total matches:", len(matches_men))

Total matches: 358


In [ ]:
matches_men = matches_men.rename(columns={
    "team1": "team_a",
    "team2": "team_b",
    "goals1": "score_a",
    "goals2": "score_b",
    "pen1": "penalty1",
    "pen2": "penalty2"
})

matches_men["date"] = pd.to_datetime(
    matches_men["date"]
)

matches_men["total_goals"] = (
    matches_men["score_a"] +
    matches_men["score_b"]
)

matches_men["goal_difference"] = (
    matches_men["score_a"] -
    matches_men["score_b"]
)

display(matches_men.head())

,date,tournament,year,round,team_a,team_b,score_a,score_b,ft_goals1,ft_goals2,penalty1,penalty2,result,group,ground,total_goals,goal_difference
0,2010-06-11,World Cup,2010,Matchday 1,South Africa,Mexico,1,1,1,1,NaN,NaN,Draw,Group A,"Soccer City, Johannesburg",2,0
1,2010-06-11,World Cup,2010,Matchday 1,Uruguay,France,0,0,0,0,NaN,NaN,Draw,Group A,"Cape Town Stadium, Cape Town",0,0
2,2010-06-16,World Cup,2010,Matchday 6,South Africa,Uruguay,0,3,0,3,NaN,NaN,Team B Win,Group A,"Loftus Versfeld Stadium, Pretoria",3,-3
3,2010-06-17,World Cup,2010,Matchday 7,France,Mexico,0,2,0,2,NaN,NaN,Team B Win,Group A,"Peter Mokaba Stadium, Polokwane",2,-2
4,2010-06-22,World Cup,2010,Matchday 12,Mexico,Uruguay,0,1,0,1,NaN,NaN,Team B Win,Group A,"Royal Bafokeng Stadium, Rustenburg",1,-1


In [ ]:
def get_result(row):

    if row["score_a"] > row["score_b"]:
        return "Team A Win"

    elif row["score_a"] < row["score_b"]:
        return "Team B Win"

    else:
        return "Draw"


matches_men["result"] = matches_men.apply(
    get_result,
    axis=1
)

matches_men["result"].value_counts()

,count
result,
Team A Win,142
Team B Win,130
Draw,86


In [ ]:
shootouts = matches_men[
    matches_men["penalty1"].notna()
].copy()

print("Matches decided by penalty shootout:", len(shootouts))

display(
    shootouts[
        [
            "date",
            "tournament",
            "year",
            "team_a",
            "team_b",
            "score_a",
            "score_b",
            "penalty1",
            "penalty2"
        ]
    ]
)

Matches decided by penalty shootout: 22


,date,tournament,year,team_a,team_b,score_a,score_b,penalty1,penalty2
54,2010-06-29,World Cup,2010,Paraguay,Japan,0,0,5.0,3.0
57,2010-07-02,World Cup,2010,Uruguay,Ghana,1,1,4.0,2.0
112,2014-06-28,World Cup,2014,Brazil,Chile,1,1,3.0,2.0
115,2014-06-29,World Cup,2014,Costa Rica,Greece,1,1,5.0,3.0
123,2014-07-05,World Cup,2014,Netherlands,Costa Rica,0,0,4.0,3.0
125,2014-07-09,World Cup,2014,Netherlands,Argentina,0,0,2.0,4.0
178,2018-07-01,World Cup,2018,Spain,Russia,1,1,3.0,4.0
179,2018-07-01,World Cup,2018,Croatia,Denmark,1,1,3.0,2.0
183,2018-07-03,World Cup,2018,Colombia,England,1,1,3.0,4.0
187,2018-07-07,World Cup,2018,Russia,Croatia,2,2,3.0,4.0


In [ ]:
print("=" * 50)
print("DATASET VALIDATION")
print("=" * 50)

print("\nTotal matches:", len(matches_men))

print("\nMatches by tournament and year:")
print(
    matches_men
    .groupby(["tournament", "year"])
    .size()
)

print("\nTournament totals:")
print(
    matches_men
    .groupby("tournament")
    .size()
)

print("\nExpected:")
print("World Cups: 256")
print("European Championships: 102")
print("Total: 358")

print("\nMissing values:")
print("Missing team_a:", matches_men["team_a"].isna().sum())
print("Missing team_b:", matches_men["team_b"].isna().sum())
print("Missing score_a:", matches_men["score_a"].isna().sum())
print("Missing score_b:", matches_men["score_b"].isna().sum())

print("\nPenalty shootouts:", matches_men["penalty1"].notna().sum())

print("\nFinal columns:")
print(matches_men.columns.tolist())

DATASET VALIDATION

Total matches: 358

Matches by tournament and year:
tournament             year
European Championship  2020    51
                       2024    51
World Cup              2010    64
                       2014    64
                       2018    64
                       2022    64
dtype: int64

Tournament totals:
tournament
European Championship    102
World Cup                256
dtype: int64

Expected:
World Cups: 256
European Championships: 102
Total: 358

Missing values:
Missing team_a: 0
Missing team_b: 0
Missing score_a: 0
Missing score_b: 0

Penalty shootouts: 22

Final columns:
['date', 'tournament', 'year', 'round', 'team_a', 'team_b', 'score_a', 'score_b', 'ft_goals1', 'ft_goals2', 'penalty1', 'penalty2', 'result', 'group', 'ground', 'total_goals', 'goal_difference']


## Dataset Validation

The final men's dataset contains 358 completed international tournament matches:

- 256 FIFA World Cup matches from 2010, 2014, 2018, and 2022.
- 102 UEFA European Championship matches from Euro 2020 and Euro 2024.

This reproduces the historical match count used by the original prediction framework.

The 2026 FIFA World Cup is intentionally excluded from this dataset. It will be treated as an independent out-of-sample test set in the model evaluation stage.

Penalty shootouts are also preserved separately from regulation/extra-time goals. This distinction is important because the Poisson model predicts goals scored during the match, while tournament simulations must determine which team advances following a shootout.

In [ ]:
matches_men.to_csv(
    "matches_men_clean.csv",
    index=False
)

print("Saved:", matches_men.shape)

Saved: (358, 17)
